In [5]:
import copy

backtrack_calls = 0
failures = 0

# =========================
# READ SUDOKU FROM FILE
# =========================
def read_sudoku(filename):
    board = []
    with open(filename, 'r') as f:
        for line in f:
            row = [int(x) for x in line.strip()]
            board.append(row)
    return board

# =========================
# PRINT BOARD
# =========================
def print_board(board):
    for row in board:
        print(" ".join(str(x) for x in row))

# =========================
# INITIAL DOMAINS
# =========================
def get_domains(board):
    domains = {}
    for r in range(9):
        for c in range(9):
            if board[r][c] == 0:
                domains[(r, c)] = set(range(1, 10))
            else:
                domains[(r, c)] = {board[r][c]}
    return domains

# =========================
# NEIGHBOURS
# =========================
def get_neighbours(cell):
    r, c = cell
    neighbours = set()

    # Row & Column
    for i in range(9):
        neighbours.add((r, i))
        neighbours.add((i, c))

    # 3x3 Box
    box_r = (r // 3) * 3
    box_c = (c // 3) * 3

    for i in range(3):
        for j in range(3):
            neighbours.add((box_r + i, box_c + j))

    neighbours.remove(cell)
    return neighbours

# =========================
# AC-3 ALGORITHM
# =========================
def ac3(domains):
    queue = [(xi, xj) for xi in domains for xj in get_neighbours(xi)]

    while queue:
        xi, xj = queue.pop(0)

        if revise(domains, xi, xj):
            if len(domains[xi]) == 0:
                return False

            for xk in get_neighbours(xi):
                if xk != xj:
                    queue.append((xk, xi))
    return True

def revise(domains, xi, xj):
    revised = False

    # Only remove if neighbor has single value
    if len(domains[xj]) == 1:
        val = next(iter(domains[xj]))
        if val in domains[xi]:
            domains[xi].remove(val)
            revised = True

    return revised

# =========================
# MRV HEURISTIC
# =========================
def select_unassigned(domains):
    unassigned = [v for v in domains if len(domains[v]) > 1]
    return min(unassigned, key=lambda v: len(domains[v])) if unassigned else None

# =========================
# BACKTRACKING + FC + AC3
# =========================
def backtrack(domains):
    global backtrack_calls, failures
    backtrack_calls += 1

    # Goal check
    if all(len(domains[v]) == 1 for v in domains):
        return domains

    var = select_unassigned(domains)

    for value in sorted(domains[var]):

        # Copy domains
        new_domains = copy.deepcopy(domains)

        # Assign value
        new_domains[var] = {value}

        # -------- Forward Checking --------
        failed = False
        for neighbor in get_neighbours(var):
            if value in new_domains[neighbor]:
                new_domains[neighbor].remove(value)
                if len(new_domains[neighbor]) == 0:
                    failed = True
                    break

        if failed:
            continue

        # -------- AC-3 --------
        if not ac3(new_domains):
            continue

        # Recursive call
        result = backtrack(new_domains)
        if result:
            return result

    failures += 1
    return None

# =========================
# SOLVER FUNCTION
# =========================
def solve_sudoku(board):
    global backtrack_calls, failures

    backtrack_calls = 0
    failures = 0

    domains = get_domains(board)

    if not ac3(domains):
        return None

    result = backtrack(domains)

    if result:
        solved = [[0]*9 for _ in range(9)]

        for (r, c), val in result.items():
            solved[r][c] = list(val)[0]

        return solved

    return None

# =========================
# MAIN FUNCTION
# =========================
if __name__ == "__main__":

    print("Select Sudoku File:")
    print("1. Easy")
    print("2. Medium")
    print("3. Hard")
    print("4. Very Hard")

    choice = input("Enter choice (1-4): ")

    files = {
        "1": "easy.txt",
        "2": "medium.txt",
        "3": "hard.txt",
        "4": "veryhard.txt"
    }

    filename = files.get(choice)

    if not filename:
        print("Invalid choice!")
        exit()

    board = read_sudoku(filename)

    print("\nInput:")
    print_board(board)

    solution = solve_sudoku(board)

    print("\nSolution:")
    if solution:
        print_board(solution)
    else:
        print("No solution found")

    print("\nBacktrack Calls:", backtrack_calls)
    print("Failures:", failures)

Select Sudoku File:
1. Easy
2. Medium
3. Hard
4. Very Hard


Enter choice (1-4):  4



Input:
0 0 1 0 0 7 0 0 0
6 0 0 4 0 0 3 0 0
0 0 0 0 3 0 0 6 4
3 8 0 0 7 6 0 0 0
0 0 0 0 0 0 0 3 6
2 7 0 0 1 5 0 0 0
0 0 0 0 2 0 0 5 1
7 0 0 1 0 0 2 0 0
0 0 8 0 0 9 0 0 0

Solution:
4 3 1 8 6 7 9 2 5
6 5 2 4 9 1 3 8 7
8 9 7 5 3 2 1 6 4
3 8 4 9 7 6 5 1 2
5 1 9 2 8 4 7 3 6
2 7 6 3 1 5 8 4 9
9 4 3 7 2 8 6 5 1
7 6 5 1 4 3 2 9 8
1 2 8 6 5 9 4 7 3

Backtrack Calls: 56
Failures: 43
